# ReviewLens: E-Commerce Review Sentiment Intelligence Dashboard
## Phase 1: Project Foundation & Raw Dataset Pipeline

**Author:** ReviewLens Senior Data Science & NLP Architecture Team  
**Environment:** Python 3.12 | Windows 11 | JupyterLab Desktop  
**Data Mode:** `DEMO` / Deterministic Synthetic Demonstration Dataset

## 1. Project Overview & Business Problem

### Business Context
E-commerce platforms receive large volumes of customer feedback across diverse product categories. Product development teams, merchants, and operations analysts need a reliable, automated pipeline to transform unstructured review text into measurable sentiment signals, competitive intelligence, and prioritized action items.

### The Long-Term ReviewLens Architecture
The long-term ReviewLens platform will support:
- Review ingestion from permitted and public sources.
- Text cleaning, sanitization, and data validation.
- Rule-based (VADER) and transformer-based sentiment scoring.
- Sentiment distributions, category benchmarking, and temporal trends.
- Production API access through FastAPI.
- Interactive visualization and executive decision-making through Streamlit.

The current notebook focuses strictly on establishing a clean, reproducible, local dataset foundation.

## 2. Project Goals & Phase 1 Scope
1. Define a strict, production-ready schema contract for raw review records.
2. Establish typed validation utilities in `backend/validators.py`.
3. Discover existing raw datasets or generate a high-quality deterministic demo dataset (`RANDOM_SEED = 42`).
4. Perform an initial data quality audit covering missingness, duplicates, ratings, and text lengths.
5. Export raw data to `data/raw/reviews_raw.csv`, quality audit to `output/tables/`, and a markdown profile to `output/reports/`.

## 3. Imports and Environment Validation

In [7]:
import sys
from pathlib import Path
import datetime
import hashlib
import importlib.metadata
import json
import logging
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import vaderSentiment

print(f'Python Version       : {sys.version.split()[0]}')
print(f'Pandas Version       : {pd.__version__}')
print(f'NumPy Version        : {np.__version__}')
print(f'vaderSentiment Ver   : {importlib.metadata.version("vaderSentiment")}')
print('Environment validation successful.')

Python Version       : 3.12.13
Pandas Version       : 3.0.5
NumPy Version        : 2.5.2
vaderSentiment Ver   : 3.3.2
Environment validation successful.


## 4. Project Path Configuration

In [8]:
# Dynamically resolve project root using pathlib
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

DATA_RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_TABLES_DIR = PROJECT_ROOT / 'output' / 'tables'
OUTPUT_REPORTS_DIR = PROJECT_ROOT / 'output' / 'reports'
OUTPUT_CHARTS_DIR = PROJECT_ROOT / 'output' / 'charts'

for folder in [DATA_RAW_DIR, DATA_PROCESSED_DIR, OUTPUT_TABLES_DIR, OUTPUT_REPORTS_DIR, OUTPUT_CHARTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root dynamically resolved to: {PROJECT_ROOT}')

Project root dynamically resolved to: C:\Users\RANADEEP\Documents\reviewlens-sentiment-dashboard


## 5. Dataset Schema Specification

The raw dataset adheres to the following production schema:

| Field Name | Type | Description |
| :--- | :--- | :--- |
| `review_id` | String | Unique identifier (e.g. `DEMO-00001`) |
| `product_name` | String | Name of the product |
| `product_category` | String | Category (Electronics, Home & Kitchen, etc.) |
| `source` | String | Source label (`Demo Dataset` or permitted source) |
| `source_url` | String | Source URL placeholder |
| `review_title` | String | Short headline of the review |
| `review_text` | String | Full body of the customer review |
| `rating` | Numeric | Star rating from 1 to 5 |
| `review_date` | ISO Date | Date of review publication (YYYY-MM-DD) |
| `verified_purchase` | Boolean | Verification flag |
| `helpful_votes` | Integer | Count of helpful votes received |
| `ingestion_timestamp` | ISO Timestamp | Pipeline ingestion timestamp |
| `data_mode` | String | Flag indicating `DEMO`, `LOCAL`, or `PERMITTED_SOURCE` |

## 6. Raw Dataset Discovery & Synthetic Generation

In [9]:
from backend.validators import RAW_SCHEMA_COLUMNS, validate_dataset_schema
from backend.data_generator import generate_synthetic_reviews, RANDOM_SEED

raw_data_path = DATA_RAW_DIR / 'reviews_raw.csv'

if raw_data_path.exists():
    print(f'Found existing raw dataset: {raw_data_path}')
    df_raw = pd.read_csv(raw_data_path)
else:
    print(f'Generating deterministic demonstration dataset (seed={RANDOM_SEED})...')
    df_raw = generate_synthetic_reviews(target_count=210, seed=RANDOM_SEED)
    df_raw.to_csv(raw_data_path, index=False)
    print(f'Saved raw dataset to: {raw_data_path}')

schema_audit = validate_dataset_schema(df_raw)
print(f'Dataset Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print(f'Schema Validation: is_valid = {schema_audit["is_valid"]}')
print(f'Columns Present ({len(schema_audit["present_columns"])}): {schema_audit["present_columns"]}')

Found existing raw dataset: C:\Users\RANADEEP\Documents\reviewlens-sentiment-dashboard\data\raw\reviews_raw.csv
Dataset Shape: 212 rows x 13 columns
Schema Validation: is_valid = True
Columns Present (13): ['review_id', 'product_name', 'product_category', 'source', 'source_url', 'review_title', 'review_text', 'rating', 'review_date', 'verified_purchase', 'helpful_votes', 'ingestion_timestamp', 'data_mode']


## 7. Raw Dataset Sample Inspection

In [10]:
print('First 10 Rows:')
display(df_raw.head(10))

print('Last 5 Rows:')
display(df_raw.tail(5))

First 10 Rows:


,review_id,product_name,product_category,source,source_url,review_title,review_text,rating,review_date,verified_purchase,helpful_votes,ingestion_timestamp,data_mode
0,DEMO-00001,UltraClean Noise-Cancelling Headphones,Electronics,Demo Dataset,https://example.com/demo-review-data,Mediocre experience,Regarding this UltraClean Noise-Cancelling Hea...,2.0,2026-02-11,False,0.0,2026-09-14T10:00:00Z,DEMO
1,DEMO-00002,AromaBreeze Stainless Steel Espresso Maker,Home and Kitchen,Demo Dataset,https://example.com/demo-review-data,Difficult setup but great once working,Purchased the AromaBreeze Stainless Steel Espr...,4.0,2025-10-05,True,0.0,2026-09-14T10:00:00Z,DEMO
2,DEMO-00003,TrailBlazer Lightweight Trekking Poles,Sports and Outdoors,Demo Dataset,https://example.com/demo-review-data,Dependable and practical,We needed a reliable solution and chose the Tr...,4.0,2025-11-11,False,4.0,2026-09-14T10:00:00Z,DEMO
3,DEMO-00004,UltraClean Noise-Cancelling Headphones,Electronics,Demo Dataset,https://example.com/demo-review-data,Very solid product with minor flaws,Regarding this UltraClean Noise-Cancelling Hea...,4.0,2025-08-25,True,0.0,2026-09-14T10:00:00Z,DEMO
4,DEMO-00005,PureAir HEPA Room Air Purifier,Home and Kitchen,Demo Dataset,https://example.com/demo-review-data,Somewhat underwhelming,Here is my honest take on the PureAir HEPA Roo...,2.0,2025-06-30,True,8.0,2026-09-14T10:00:00Z,DEMO
5,DEMO-00006,PureAir HEPA Room Air Purifier,Home and Kitchen,Demo Dataset,https://example.com/demo-review-data,NaN,"Regarding this PureAir HEPA Room Air Purifier,...",1.0,2026-01-28,True,0.0,2026-09-14T10:00:00Z,DEMO
6,DEMO-00007,Designing Data-Intensive Applications,Books,Demo Dataset,https://example.com/demo-review-data,Satisfied with this buy,I have been testing this Designing Data-Intens...,4.0,2025-11-09,True,3.0,2026-09-14T10:00:00Z,DEMO
7,DEMO-00008,PureAir HEPA Room Air Purifier,Home and Kitchen,Demo Dataset,https://example.com/demo-review-data,"Loved the design, hated the durability",Having ordered the PureAir HEPA Room Air Purif...,2.0,2025-02-10,True,1.0,2026-09-14T10:00:00Z,DEMO
8,DEMO-00009,HydraGlow Hyaluronic Acid Facial Serum,Beauty and Personal Care,Demo Dataset,https://example.com/demo-review-data,Best purchase of the year,Having ordered the HydraGlow Hyaluronic Acid F...,5.0,2025-10-19,True,NaN,2026-09-14T10:00:00Z,DEMO
9,DEMO-00010,UltraClean Noise-Cancelling Headphones,Electronics,Demo Dataset,https://example.com/demo-review-data,Ordinary product without highlights,Purchased the UltraClean Noise-Cancelling Head...,3.0,2025-04-02,True,2.0,2026-09-14T10:00:00Z,DEMO


Last 5 Rows:


,review_id,product_name,product_category,source,source_url,review_title,review_text,rating,review_date,verified_purchase,helpful_votes,ingestion_timestamp,data_mode
207,DEMO-00208,ChefMaster 8-Piece Japanese Knife Set,Home and Kitchen,Demo Dataset,https://example.com/demo-review-data,Great hardware but dreadful software,"After reading mixed feedback online, I tried t...",3.0,2025-05-17,True,5.0,2026-09-14T10:00:00Z,DEMO
208,DEMO-00209,HydroFlow Insulated Stainless Water Bottle,Sports and Outdoors,Demo Dataset,https://example.com/demo-review-data,Good quality for the price,Regarding this HydroFlow Insulated Stainless W...,4.0,2026-01-25,True,1.0,2026-09-14T10:00:00Z,DEMO
209,DEMO-00210,PureAir HEPA Room Air Purifier,Home and Kitchen,Demo Dataset,https://example.com/demo-review-data,Ordinary product without highlights,<div><b>Ordinary product without highlights</b...,3.0,2025-07-25,False,0.0,2026-09-14T10:00:00Z,DEMO
210,DEMO-00015,TrailBlazer Lightweight Trekking Poles,Sports and Outdoors,Demo Dataset,https://example.com/demo-review-data,Absolutely phenomenal quality!,I have been testing this TrailBlazer Lightweig...,5.0,2025-10-05,True,3.0,2026-09-14T10:00:00Z,DEMO
211,DEMO-00029,ApexFit Resistance Band Training Set,Sports and Outdoors,Demo Dataset,https://example.com/demo-review-data,Ordinary product without highlights,Having ordered the ApexFit Resistance Band Tra...,3.0,2025-04-09,True,0.0,2026-09-14T10:00:00Z,DEMO


## 8. Initial Data Quality Report

In [11]:
from backend.pipeline import generate_raw_quality_report, generate_raw_dataset_profile_md

quality_report_df = generate_raw_quality_report(df_raw)
display(quality_report_df)

quality_report_csv = OUTPUT_TABLES_DIR / 'raw_data_quality_report.csv'
quality_report_df.to_csv(quality_report_csv, index=False)
print(f'Saved quality audit table to: {quality_report_csv}')

profile_md = generate_raw_dataset_profile_md(df_raw, quality_report_df)
profile_report_path = OUTPUT_REPORTS_DIR / 'raw_dataset_profile.md'
profile_report_path.write_text(profile_md, encoding='utf-8')
print(f'Saved raw dataset markdown profile to: {profile_report_path}')

,Metric,Value,Description
0,Total Rows,212,Total raw review records ingested
1,Total Columns,13,Total schema columns
2,Fully Duplicate Rows,2,Identical rows across all fields
3,Duplicate Review Texts,4,Rows sharing identical review text
4,Missing Review Titles,3 (1.4%),Records missing a review title
5,Missing Helpful Votes,3 (1.4%),Records missing helpful vote count
6,Invalid Ratings Count,4,Ratings outside 1-5 integer scale
7,Unique Products,15,Distinct product names represented
8,Unique Categories,5,Distinct product categories
9,Earliest Review Date,2025-01-15,Minimum review timestamp


Saved quality audit table to: C:\Users\RANADEEP\Documents\reviewlens-sentiment-dashboard\output\tables\raw_data_quality_report.csv
Saved raw dataset markdown profile to: C:\Users\RANADEEP\Documents\reviewlens-sentiment-dashboard\output\reports\raw_dataset_profile.md


## 9. Baseline Distributions & Descriptive Statistics

In [12]:
print('--- Rating Distribution (including anomalies) ---')
print(df_raw['rating'].value_counts(dropna=False).sort_index())

print('\n--- Product Category Distribution ---')
print(df_raw['product_category'].value_counts())

print('\n--- Source Distribution ---')
print(df_raw['source'].value_counts())

print('\n--- Verified Purchase Distribution ---')
print(df_raw['verified_purchase'].value_counts())

print('\n--- Helpful Votes Descriptive Statistics ---')
print(pd.to_numeric(df_raw['helpful_votes'], errors='coerce').describe())

word_counts = df_raw['review_text'].dropna().apply(lambda t: len(str(t).split()))
print('\n--- Review Word Count Descriptive Statistics ---')
print(word_counts.describe())

date_col = pd.to_datetime(df_raw['review_date'], errors='coerce')
print(f'\nUnique Products   : {df_raw["product_name"].nunique()}')
print(f'Unique Categories : {df_raw["product_category"].nunique()}')
print(f'Earliest Review   : {date_col.min().strftime("%Y-%m-%d")}')
print(f'Latest Review     : {date_col.max().strftime("%Y-%m-%d")}')

--- Rating Distribution (including anomalies) ---
rating
-1.0     1
 0.0     1
 1.0    16
 2.0    34
 3.0    33
 4.0    50
 5.0    75
 6.0     1
 NaN     1
Name: count, dtype: int64

--- Product Category Distribution ---
product_category
Home and Kitchen            52
Electronics                 41
Beauty and Personal Care    41
Books                       40
Sports and Outdoors         38
Name: count, dtype: int64

--- Source Distribution ---
source
Demo Dataset    212
Name: count, dtype: int64

--- Verified Purchase Distribution ---
verified_purchase
True     173
False     39
Name: count, dtype: int64

--- Helpful Votes Descriptive Statistics ---
count    209.000000
mean       1.301435
std        1.906479
min        0.000000
25%        0.000000
50%        0.000000
75%        2.000000
max       11.000000
Name: helpful_votes, dtype: float64

--- Review Word Count Descriptive Statistics ---
count    211.000000
mean      31.236967
std        4.818690
min        0.000000
25%       29.0000

## 10. Dataset Export and Reproducibility Summary

The data foundation pipeline has verified and exported:
1. `data/raw/reviews_raw.csv` — Full raw ingested demonstration dataset.
2. `output/tables/raw_data_quality_report.csv` — Comprehensive tabular quality metrics.
3. `output/reports/raw_dataset_profile.md` — Formatted Markdown summary profile.

## 11. Next Steps
The dataset foundation is ready for Phase 2 processing in `notebooks/02_sentiment_analysis_and_eda.ipynb`, which encompasses:
- Text sanitization (whitespace normalization, HTML tag removal, URL tokenization).
- Deduplication and removal of invalid ratings.
- Feature engineering (NLP metrics, temporal attributes, rating groups).
- VADER sentiment scoring and intensity segmentation.
- Exploratory data analysis, 10 high-resolution charts, and business intelligence reporting.